# 01 - Data Generation (Kaggle)
Build MedScript instruction pairs from MedDialog and synthetic SOAP labels.

In [ ]:
from datasets import load_dataset
import pandas as pd

# Define it here since this is a separate notebook
SYSTEM_PROMPT = "You are a clinical assistant. Given an unstructured doctor's note, generate a structured SOAP summary."

def build_chat_text(note: str, soap: str) -> str:
    return (
        "<|im_start|>system\n"
        f"{SYSTEM_PROMPT}<|im_end|>\n"
        "<|im_start|>user\n"
        f"{note}<|im_end|>\n"
        "<|im_start|>assistant\n"
        f"{soap}<|im_end|>"
    )

ds = load_dataset('omi-health/medical-dialogue-to-soap-summary')

train_df = ds['train'].to_pandas()[['dialogue', 'soap']]
eval_df = ds['validation'].to_pandas()[['dialogue', 'soap']]

train_df = train_df.rename(columns={'dialogue': 'raw_note'})
eval_df = eval_df.rename(columns={'dialogue': 'raw_note'})

train_df['text'] = train_df.apply(lambda r: build_chat_text(r['raw_note'], r['soap']), axis=1)
eval_df['text'] = eval_df.apply(lambda r: build_chat_text(r['raw_note'], r['soap']), axis=1)

train_df.to_json('/kaggle/working/medscript_train.jsonl', orient='records', lines=True)
eval_df.to_json('/kaggle/working/medscript_eval.jsonl', orient='records', lines=True)

print(f"Train: {len(train_df)}, Eval: {len(eval_df)}")

In [3]:
%%capture
!pip install uv
!uv pip install --system unsloth unsloth_zoo wandb
!uv pip install --system --reinstall transformers

In [4]:
import os
import torch

# Unsloth MUST be first
from unsloth import FastLanguageModel

# Then everything else
from datasets import load_dataset
from trl import SFTTrainer, SFTConfig

In [5]:
from kaggle_secrets import UserSecretsClient
secrets = UserSecretsClient()

import os
os.environ['WANDB_API_KEY'] = secrets.get_secret("WANDB_API_KEY")
os.environ['HF_TOKEN'] = secrets.get_secret("HF_TOKEN")

import wandb
wandb.init(project="medscript", name="qwen2.5-3b-qlora")

wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from WANDB_API_KEY.
wandb: Currently logged in as: ravikantsaraf03 (ravikantsaraf03-dayananda-sagar-institutions) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


wandb: Detected [huggingface_hub.inference, openai] in use.
wandb: Use W&B Weave for improved LLM call tracing. Install Weave with `pip install weave` then add `import weave` to the top of your script.
wandb: For more information, check out the docs at: https://weave-docs.wandb.ai/


In [7]:
train_ds = load_dataset('json', data_files='/kaggle/working/medscript_train.jsonl', split='train')
eval_ds = load_dataset('json', data_files='/kaggle/working/medscript_eval.jsonl', split='train')
train_ds, eval_ds

Generating train split: 0 examples [00:00, ? examples/s]

Generating train split: 0 examples [00:00, ? examples/s]

(Dataset({
     features: ['raw_note', 'soap', 'text'],
     num_rows: 9250
 }),
 Dataset({
     features: ['raw_note', 'soap', 'text'],
     num_rows: 500
 }))

In [24]:
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name='unsloth/Qwen2.5-3B-Instruct',
    max_seq_length=2048,
    dtype=None,
    load_in_4bit=True,
)

model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    lora_alpha=32,
    target_modules=['q_proj', 'k_proj', 'v_proj', 'o_proj', 'gate_proj', 'up_proj', 'down_proj'],
    lora_dropout=0,
    bias='none',
    use_gradient_checkpointing='unsloth',
    random_state=42,
)

==((====))==  Unsloth 2026.4.6: Fast Qwen2 patching. Transformers: 5.5.4.
   \\   /|    Tesla T4. Num GPUs = 2. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

unsloth/qwen2.5-3b-instruct-unsloth-bnb-4bit does not have a padding token! Will use pad_token = <|PAD_TOKEN|>.


Unsloth 2026.4.6 patched 36 layers with 36 QKV layers, 36 O layers and 36 MLP layers.


In [26]:
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

In [27]:
# Change SFTConfig — remove max_seq_length from here:
cfg = SFTConfig(
    output_dir='/kaggle/working/medscript_ckpt',
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,
    num_train_epochs=3,
    max_steps=1000,           # ← ADDED THIS — Due to GPU Constraints
    learning_rate=2e-4,
    lr_scheduler_type='cosine',
    warmup_ratio=0.03,
    fp16=not torch.cuda.is_bf16_supported(),
    bf16=torch.cuda.is_bf16_supported(),
    logging_steps=10,
    eval_strategy='steps',
    eval_steps=100,
    save_strategy='steps',   # ← change epoch to steps
    save_steps=250,
    report_to='wandb',
)

# And pass max_seq_length directly to SFTTrainer:
trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=train_ds,
    eval_dataset=eval_ds,
    dataset_text_field='text',
    max_seq_length=2048,        
    args=cfg,
)

warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


Unsloth: Tokenizing ["text"] (num_proc=8):   0%|          | 0/9250 [00:00<?, ? examples/s]

Unsloth: Tokenizing ["text"] (num_proc=8):   0%|          | 0/500 [00:00<?, ? examples/s]

🦥 Unsloth: Padding-free auto-enabled, enabling faster training.


In [28]:
trainer.train()

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None, 'pad_token_id': 151645}.
==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 9,250 | Num Epochs = 1 | Total steps = 1,000
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 4 x 1) = 8
 "-____-"     Trainable parameters = 29,933,568 of 3,115,872,256 (0.96% trained)


Step,Training Loss,Validation Loss
100,1.115837,1.109177
200,1.100867,1.082709
300,1.086232,1.066665
400,1.039215,1.053844
500,1.036002,1.045907
600,1.036266,1.038836
700,1.049151,1.033528
800,1.029365,1.029485
900,1.045415,1.027262
1000,1.051148,1.026900


TrainOutput(global_step=1000, training_loss=1.068920569419861, metrics={'train_runtime': 13355.5188, 'train_samples_per_second': 0.599, 'train_steps_per_second': 0.075, 'total_flos': 1.3483896365614694e+17, 'train_loss': 1.068920569419861, 'epoch': 0.8648648648648649})

In [29]:
trainer.model.save_pretrained('/kaggle/working/medscript_adapters')
tokenizer.save_pretrained('/kaggle/working/medscript_adapters')
print('Saved adapters to /kaggle/working/medscript_adapters')

Saved adapters to /kaggle/working/medscript_adapters


In [30]:
# Merge LoRA into base model for faster inference
FastLanguageModel.for_inference(model)

# Test with a sample clinical note
test_note = """
Patient is a 45-year-old male presenting with chest pain for the past 2 hours. 
Pain is described as pressure-like, radiating to left arm. 
Patient has history of hypertension and is currently on lisinopril. 
Blood pressure 150/90, heart rate 98, oxygen saturation 97%. 
ECG shows ST elevation in leads II, III, aVF.
"""

inputs = tokenizer(
    f"<|im_start|>system\nYou are a clinical assistant. Given an unstructured doctor's note, generate a structured SOAP summary.<|im_end|>\n<|im_start|>user\n{test_note}<|im_end|>\n<|im_start|>assistant\n",
    return_tensors="pt"
).to("cuda")

outputs = model.generate(
    **inputs,
    max_new_tokens=512,
    temperature=0.1,
    do_sample=True,
)

response = tokenizer.decode(outputs[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True)
print("=== FINE-TUNED MODEL OUTPUT ===")
print(response)

Both `max_new_tokens` (=512) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:71: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:281: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)


=== FINE-TUNED MODEL OUTPUT ===
S: The patient is a 45-year-old male who presents with chest pain that started two hours ago, described as pressure-like and radiating to the left arm. He has a history of hypertension and is currently on lisinopril. He reports no other symptoms or relevant medical history.
O: Vital signs include blood pressure at 150/90 mmHg, heart rate at 98 bpm, and oxygen saturation at 97% on room air. An electrocardiogram (ECG) reveals ST elevation in leads II, III, and aVF.
A: The primary assessment is acute myocardial infarction (AMI), evidenced by ST-segment elevation on ECG and typical symptoms of chest pain. Differential diagnoses could include stable angina or a non-ST elevation myocardial infarction (NSTEMI), but the presence of ST elevation strongly suggests AMI.
P: Immediate management includes administration of aspirin, clopidogrel, and nitroglycerin sublingual tablets. The patient will be transferred to the cardiac catheterization laboratory for further e

In [31]:
# Merge adapters into base model and save as full model
model.save_pretrained_merged(
    '/kaggle/working/medscript_full',
    tokenizer,
    save_method='merged_16bit'  # saves full merged model in 16bit
)

config.json:   0%|          | 0.00/757 [00:00<?, ?B/s]

Found HuggingFace hub cache directory: /root/.cache/huggingface/hub


Fetching 1 files:   0%|          | 0/1 [00:00<?, ?it/s]

Checking cache directory for required files...
Cache check failed: model-00001-of-00002.safetensors not found in local cache.
Not all required files found in cache. Will proceed with downloading.
Checking cache directory for required files...
Cache check failed: tokenizer.model not found in local cache.
Not all required files found in cache. Will proceed with downloading.



Unsloth: Preparing safetensor model files:   0%|          | 0/2 [00:00<?, ?it/s]

...

model-00002-of-00002.safetensors:   0%|          | 0.00/2.20G [00:00<?, ?B/s]


Unsloth: Preparing safetensor model files: 100%|██████████| 2/2 [00:15<00:00,  7.98s/it]


Note: tokenizer.model not found (this is OK for non-SentencePiece models)



Unsloth: Merging weights into 16bit: 100%|██████████| 2/2 [00:44<00:00, 22.41s/it]


Unsloth: Merge process complete. Saved to `/kaggle/working/medscript_full`


In [ ]:
#For Adaptors

import shutil

# Zip the adapters folder (small, ~50MB)
shutil.make_archive('/kaggle/working/medscript_adapters', 'zip', '/kaggle/working/medscript_adapters')
print("Zipped! Find medscript_adapters.zip in the Output tab")

In [35]:
from huggingface_hub import login
from kaggle_secrets import UserSecretsClient

# Login
secrets = UserSecretsClient()
login(token=secrets.get_secret("HF_TOKEN"))

# Push adapters only (recommended — lightweight, ~50MB)
model.push_to_hub(
    "Ravi2003/medscript-qwen2.5-3b-qlora",
    token=secrets.get_secret("HF_TOKEN")
)
tokenizer.push_to_hub(
    "Ravi2003/medscript-qwen2.5-3b-qlora",
    token=secrets.get_secret("HF_TOKEN")
)
print("Pushed to HF Hub!")

README.md:   0%|          | 0.00/578 [00:00<?, ?B/s]

Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

Saved model to https://huggingface.co/Ravi2003/medscript-qwen2.5-3b-qlora


Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

Pushed to HF Hub!


## Evaluation

In [36]:
!pip install -q rouge-score bert-score

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.1/61.1 kB 2.0 MB/s eta 0:00:00


In [43]:
from datasets import load_dataset
from rouge_score import rouge_scorer
from tqdm.auto import tqdm
import pandas as pd, json, torch

eval_ds = load_dataset('json', data_files='/kaggle/working/medscript_eval.jsonl', split='train')
eval_df = eval_ds.to_pandas().sample(n=50, random_state=42).reset_index(drop=True)

SYSTEM_PROMPT = "You are a clinical assistant. Given an unstructured doctor's note, generate a structured SOAP summary."

def generate_soap(mdl, tok, note):
    prompt = f"<|im_start|>system\n{SYSTEM_PROMPT}<|im_end|>\n<|im_start|>user\n{note}<|im_end|>\n<|im_start|>assistant\n"
    inputs = tok(prompt, return_tensors="pt").to("cuda")
    with torch.no_grad():
        out = mdl.generate(**inputs, max_new_tokens=512, temperature=0.1, do_sample=True)
    return tok.decode(out[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True)

FastLanguageModel.for_inference(model)

ft_outputs = []
for _, row in tqdm(eval_df.iterrows(), total=len(eval_df)):
    ft_outputs.append(generate_soap(model, tokenizer, row['raw_note']))
eval_df['finetuned'] = ft_outputs
print("Fine-tuned done!")

  0%|          | 0/50 [00:00<?, ?it/s]

Both `max_new_tokens` (=512) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:71: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:281: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
Both `max_new_tokens` (=5

Fine-tuned done!


In [46]:
eval_df.to_csv('/kaggle/working/eval_finetuned_only.csv', index=False)
print("Saved!")

Saved!


## For Comparision of this FInetuned Model with the base model , look at the evaluation_strategy.ipynb noteboook